<a href="https://colab.research.google.com/github/ucaokylong/Some_small_projects/blob/main/agent_smolagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install smolagents
!pip install "smolagents[transformers]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.4 MB/s eta 0:00:00


In [4]:
from smolagents import TransformersModel, CodeAgent, FinalAnswerTool, tool
import os
import torch

def set_seed(seed: int):
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed(42)

In [15]:
from smolagents import tool
import requests
from bs4 import BeautifulSoup

@tool
def fetch_latest_news_titles_and_urls(url: str) -> list[tuple[str, str]]:
    """
    This tool extracts the titles and URLs of the latest news articles from a news website's homepage and its main sections.

    Args:
        url: The base URL of the news website (e.g., 'https://vnexpress.net').

    Returns:
        A list of tuples, where each tuple contains (article_title, article_url).
    """
    import requests
    from bs4 import BeautifulSoup

    article_urls = []
    article_titles = []
    navigation_urls = []

    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        navigation_bar = soup.find("nav", class_="main-nav")
        if navigation_bar:
            for header in navigation_bar.ul.find_all("li")[2:7]:
                href = header.a["href"]
                full_url = href if href.startswith("http") else url.rstrip('/') + href
                navigation_urls.append(full_url)

        for section_url in navigation_urls:
            section_response = requests.get(section_url)
            section_soup = BeautifulSoup(section_response.text, "html.parser")

            for article in section_soup.find_all("article"):
                title_tag = article.find("h3", class_="title-news")
                if title_tag:
                    title = title_tag.text.strip()
                    link_tag = article.find("a")
                    if link_tag and link_tag.has_attr('href'):
                        article_url = link_tag["href"]
                        article_titles.append(title)
                        article_urls.append(article_url)

    except Exception as e:
        print(f"Error fetching news: {e}")

    return list(zip(article_titles, article_urls))

In [16]:
from smolagents import tool

@tool
def extract_news_article_content(url: str) -> str:
    """
    This tool extracts the content of a news article from its URL. [cite: 524]

    Args:
        url: The URL of the news article. [cite: 529]

    Returns:
        The full text content of the news article. [cite: 537]
    """
    import requests
    from bs4 import BeautifulSoup

    # Truy cập URL của bài báo [cite: 544, 556]
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    content = ""
    # Duyệt qua các thẻ <p> để lấy toàn bộ nội dung văn bản [cite: 551, 554]
    for paragraph in soup.find_all("p"):
        content += paragraph.get_text().strip() + " "

    return content.strip()

In [17]:
from smolagents import tool

@tool
def summarize_news(text: str) -> str:
    """
    This tool summarizes the given Vietnamese news text.

    Args:
        text: The Vietnamese news text to be summarized.

    Returns:
        The summarized version of the input text.
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Thiết lập thiết bị xử lý
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "VietAI/vit5-base-vietnews-summarization"

    # Khởi tạo tokenizer và model với định dạng bfloat16 để tối ưu bộ nhớ
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
    model.to(device)

    # Tiền xử lý văn bản đầu vào theo định dạng của ViT5
    formatted_text = "vietnews: " + text + " </s>"
    encoding = tokenizer(formatted_text, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_masks = encoding["attention_mask"].to(device)

    # Thực hiện quá trình tóm tắt văn bản
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_masks,
            max_length=256,
        )

    # Giải mã kết quả trả về từ mô hình
    summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [18]:
from smolagents import tool

@tool
def classify_topic(text: str, topic: str) -> bool:
    """
    This tool classifies whether the given Vietnamese text is related to the specified topic using zero-shot classification.

    Args:
        text: The Vietnamese text to be classified.
        topic: The string representing the topic to be checked.

    Returns:
        True if the text is related to the topic; False otherwise.
    """
    import torch
    from transformers import pipeline

    # Thiết lập thiết bị xử lý (GPU nếu có, ngược lại dùng CPU) [cite: 664]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Khởi tạo pipeline phân loại zero-shot với mô hình đa ngôn ngữ [cite: 665, 666]
    classifier = pipeline(
        "zero-shot-classification",
        model="vicgalle/xlm-roberta-large-xnli-anli",
        device=device,
        trust_remote_code=True,
    )

    # Định nghĩa các nhãn ứng viên dựa trên chủ đề đầu vào [cite: 653]
    candidate_labels = [topic, f"không liên quan {topic}"]

    # Thực hiện phân loại văn bản [cite: 654]
    result = classifier(text, candidate_labels)
    predicted_label = result["labels"][0] # Lấy nhãn có xác suất cao nhất [cite: 655]

    return predicted_label == topic

In [20]:
from smolagents import TransformersModel, CodeAgent, FinalAnswerTool
import torch

# 1. Khởi tạo mô hình ngôn ngữ (Model)
# Sử dụng Qwen2.5-Coder-3B-Instruct, tối ưu cho việc sinh mã Python trong CodeAgent
model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    max_new_tokens=2000
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [21]:


# 2. Khởi tạo Agent (CodeAgent)
# Kết nối mô hình với các công cụ đã định nghĩa và cho phép thực thi mã Python
agent = CodeAgent(
    model=model,
    tools=[
        fetch_latest_news_titles_and_urls, # Đã sửa tên hàm ở đây
        summarize_news,
        extract_news_article_content,
        classify_topic,
        FinalAnswerTool()
    ],
    additional_authorized_imports=["requests", "bs4"],
    verbosity_level=2,
    name="news_agent"
)

In [ ]:
agent.visualize()

In [22]:
# Định nghĩa nhiệm vụ cụ thể cho Agent
task = """
Hãy thu thập các tin tức thuộc chủ đề "trí tuệ nhân tạo" trên https://vnexpress.net.
""".strip()

# Chạy Agent và lưu kết quả trả về vào biến result
try:
    result = agent.run(task)
    print("Agent execution completed. Result:")
    print(result)
except Exception as e:
    print(f"An error occurred during agent execution: {e}")
    result = None # Or handle as appropriate

╭───────────────────────────────────────────── New run - news_agent ──────────────────────────────────────────────╮
│                                                                                                                 │
│ Hãy thu thập các tin tức thuộc chủ đề "trí tuệ nhân tạo" trên https://vnexpress.net.                            │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-3B-Instruct ────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To collect news articles about AI on the VNExpress website, I will use the                                
`fetch_latest_news_titles_and_urls` tool to retrieve the titles and URLs of the latest news articles related to AI.
Then, I will use the `extract_news_article_content` tool to extract the content of each article.                   
<code>                                                                                                             
latest_news = fetch_latest_news_titles_and_urls(url="https://vnexpress.net")                                       
print(latest_news)                                                                                                 
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_news = fetch_latest_news_titles_and_urls(url="https://vnexpress.net")                                     
  print(latest_news)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[('Lấy ý kiến xét tặng danh hiệu Anh hùng Lao động cho ông Johnathan Hạnh Nguyễn', 
'https://vnexpress.net/lay-y-kien-xet-tang-danh-hieu-anh-hung-lao-dong-cho-ong-johnathan-hanh-nguyen-5074390.html')
, ('Ôm bao tải nhận tiền đền bù đất', 'https://vnexpress.net/om-bao-tai-nhan-tien-den-bu-dat-5074303.html'), ('Cặp 
cá voi kiếm ăn ở biển Đăk Lăk', 'https://vnexpress.net/cap-ca-voi-kiem-an-o-bien-dak-lak-5074320.html'), ('Đề xuất 
thay đổi quy mô tuyến vượt biển Cần Giờ - Vũng Tàu', 
'https://vnexpress.net/de-xuat-thay-doi-quy-mo-tuyen-vuot-bien-can-gio-vung-tau-5074205.html'), ('Ông Trần Anh Tuấn
và Phan Văn Trọng làm trợ lý Thủ tướng', 
'https://vnexpress.net/ong-tran-anh-tuan-va-phan-van-trong-lam-tro-ly-thu-tuong-5074293.html'), ('Nội đô Hà Nội 
thành đại công trường', 'https://vnexpress.net/noi-do-ha-noi-thanh-dai-cong-truong-5073916.html'), ('Cháy nhà lúc 
rạng sáng ở Hà Nội, một người tử vong', 
'https://vnexpress.net/chay-nha-luc-rang-sang-o-ha-noi-mot-nguoi-tu-vong-5074254.html'), ('Hà Nội hỗ trợ người dân 
khu vực tái thiết đô thị thế nào?', 
'https://vnexpress.net/ha-noi-ho-tro-nguoi-dan-khu-vuc-tai-thiet-do-thi-the-nao-5074054.html'), ('Cá sấu nổi trong 
ao nhà dân', 'https://vnexpress.net/ca-sau-noi-trong-ao-nha-dan-5074139.html'), ('Đường Cộng Hòa thông thoáng ngày 
đầu phân luồng đảo chiều', 
'https://vnexpress.net/duong-cong-hoa-thong-thoang-ngay-dau-phan-luong-dao-chieu-5074110.html'), ("Gốm Bát Tràng: 
Thủ công độc bản hay công nghệ 'nhân bản'?", 
'https://vnexpress.net/gom-bat-trang-thu-cong-doc-ban-hay-cong-nghe-nhan-ban-5071132.html'), ('Nhà ga sân bay Long 
Thành vắng lao động giai đoạn về đích', 
'https://vnexpress.net/nha-ga-san-bay-long-thanh-vang-lao-dong-giai-doan-ve-dich-5073862.html'), ('Nơi an nghỉ của 
hai vua triều Nguyễn bị lưu đày', 
'https://vnexpress.net/noi-an-nghi-cua-hai-vua-trieu-nguyen-bi-luu-day-5073900.html'), ('Đề xuất mở rộng phạm vi 
cấm ôtô trên 16 chỗ vào Hoàn Kiếm', 
'https://vnexpress.net/de-xuat-mo-rong-pham-vi-cam-oto-tren-16-cho-vao-hoan-kiem-5074047.html'), ('TP HCM dự kiến 
chia giai đoạn xây 19 ga đường sắt đi Long Thành', 
'https://vnexpress.net/tp-hcm-du-kien-chia-giai-doan-xay-19-ga-duong-sat-di-long-thanh-5074041.html'), ("Lỗ hổng 
khiến 'trái cây tỷ đô' dễ nhiễm chất cấm", 
'https://vnexpress.net/lo-hong-khien-trai-cay-ty-do-de-nhiem-chat-cam-5073681.html'), ('Trung tâm bảo trợ xã hội 
quy mô nhất Hải Phòng', 'https://vnexpress.net/trung-tam-bao-tro-xa-hoi-quy-mo-nhat-hai-phong-5072363.html'), ('21 
tỉnh thành nắng nóng gay gắt', 'https://vnexpress.net/21-tinh-thanh-nang-nong-gay-gat-5074005.html'), ('Tổng Bí 
thư, Chủ tịch nước: Trẻ em không thể lớn lên trong chửi mắng, bạo lực', 
'https://vnexpress.net/tong-bi-thu-chu-tich-nuoc-tre-em-khong-the-lon-len-trong-chui-mang-bao-luc-5074035.html'), 
('Điều chỉnh giao thông tại vòng xoay trung tâm TP HCM', 
'https://vnexpress.net/dieu-chinh-giao-thong-tai-vong-xoay-trung-tam-tp-hcm-5074033.html'), ('Đề xuất bổ sung quy 
hoạch 3 tuyến cao tốc mới', 'https://vnexpress.net/de-xuat-bo-sung-quy-hoach-3-tuyen-cao-toc-moi-5073926.html'), 
('Loài chim quý nhất thế giới hồi hương sau hơn 100 năm', 
'https://vnexpress.net/loai-chim-quy-nhat-the-gioi-hoi-huong-sau-hon-100-nam-5073915.html'), ('Doanh nghiệp bị phạt
vì khiến nước giếng đổi màu tím bất thường', 
'https://vnexpress.net/doanh-nghiep-bi-phat-vi-khien-nuoc-gieng-doi-mau-tim-bat-thuong-5073906.html'), ("'Chấm dứt 
tình trạng hội họp nhiều, sản phẩm ít'", 
'https://vnexpress.net/cham-dut-tinh-trang-hoi-hop-nhieu-san-pham-it-5073891.html'), ('Phối cảnh chuỗi không gian 
công cộng ven sông Hồng', 'https://vnexpress.net/phoi-canh-chuoi-khong-gian-cong-cong-ven-song-hong-5073751.html'),
('Thứ trưởng Xây dựng: Năm 2027 từ TP HCM tới sân bay Long Thành mất 30 phút', 
'https://vnexpress.net/thu-truong-xay-dung-nam-2027-tu-tp-hcm-toi-san-bay-long-thanh-mat-30-phut-5073832.html'), 
('Hà Nội dự kiến hạn chế xe cá nhân từ 2035', 
'https://vnexpress.net/ha-noi-du-kien-han-ch

[Step 1: Duration 58.16 seconds| Input tokens: 2,280 | Output tokens: 749]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
CUDA out of memory. Tried to allocate 746.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 559.81 MiB is 
free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 13.47 GiB is 
allocated by PyTorch, and 422.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is 
large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory
Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

[Step 2: Duration 0.30 seconds]

AgentGenerationError: Error in generating model output:
CUDA out of memory. Tried to allocate 746.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 559.81 MiB is free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 13.47 GiB is allocated by PyTorch, and 422.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)